In [ ]:
# --- Determinism guardrails (put BEFORE importing torch-heavy libs) ---
import os
os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import random
import numpy as np
import torch
import multiprocessing as mp

try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass

BASE_SEED = 42
random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)
torch.cuda.manual_seed_all(BASE_SEED)

torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_num_threads(1)

# Optionally pin to CPU for strict reproducibility:
# device = 'cpu'

In [ ]:
# Set up the file path
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(parent_dir)

In [ ]:
# Import general packages
from openpyxl import Workbook, load_workbook
from datetime import datetime
# import torch
from matplotlib import pyplot as plt
from pytorch_lightning.loggers import CometLogger
import numpy as np
from itertools import product
from tqdm import tqdm

# Import Agent-Environment packages
from RL4CRN.environments.environment import Environment
from RL4CRN.environments.parallel_environments import ParallelEnvironments
from RL4CRN.environments.serial_environments import SerialEnvironments
from RL4CRN.agents.reinforce_agent import REINFORCEAgent
from RL4CRN.policies.add_reaction_by_index import AddReactionByIndex

# Import Interface packages
from RL4CRN.env2agent_interface.explicit_observer import ExplicitObserver
from RL4CRN.env2agent_interface.explicit_tensorizer import ExplicitTensorizer
from RL4CRN.agent2env_interface.library_actuator import LibraryActuator
from RL4CRN.agent2env_interface.iocrn_stepper import IOCRNStepper

# Import CRN packages
from RL4CRN.iocrns.iocrn import IOCRN
from RL4CRN.iocrns.reactions import MassAction
from RL4CRN.utils.ic import IC
from RL4CRN.iocrns.reaction_library import construct_mass_action_library

# Import Reward packages
from RL4CRN.rewards.deterministic import dynamic_tracking_error

In [ ]:
# Set the logger to use Comet
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
api_key = "o77J6VCMDamustkfJuMXZ2jdV"
logger = CometLogger(
    api_key=api_key,
    project="Molecular_Integrators",        
    workspace="maurice-filo", 
    name=f'Transients_Experiment_{timestamp}',
)
# logger = logger.experiment
logger = None

In [ ]:
# Construct the template CRN
r1 = MassAction(reactant_labels=[], product_labels=['Z_1'], input_channels=['u_1'], params=[1.], params_controllability=[True])
r2 = MassAction(reactant_labels=['X_1'], product_labels=[], input_channels=['u_2'], params=[1.], params_controllability=[True])
crn_template = IOCRN([r1, r2], output_labels=['X_1'])
crn_template.compile()
p = crn_template.num_inputs
print("Template CRN:")
print(crn_template)

# Construct the library of possible reactions
species_labels = ['X_1', 'Z_1', 'Z_2']
library = construct_mass_action_library(species_labels=species_labels, order=2)
crn_template.set_library_context(library)
M = len(library.reactions)
K = library.get_num_parameters()
print("Library of possible reactions:")
print(library)
print("------------------------------------------------")

In [ ]:
# Device
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
device = 'cpu'
print(f'Using device: {device}')

In [ ]:
# Flags and filenames
save_flag = False                                               # Save the agent checkpoint
load_flag = False                                               # Load the agent checkpoint 
train_flag = True                                               # Train the agent
save_sheet_flag = False                                         # Save the configuration to an Excel sheet

save_filename = timestamp + '.pth'                              # Filename for saving the agent checkpoint
load_filename = ''                                              # Filename for loading the agent checkpoint
file_name = "Discover_Molecular_Integrators_REINFORCE.xlsx"     # Filename for saving the Excel sheet

In [ ]:
# Hyperparameters
max_added_reactions = 5                             # Maximum number of reactions
N_CPUs = os.cpu_count()                             # Number of CPUs          
N = 10#10*N_CPUs                                       # Number of samples (batch size)    
width = 1024                                        # Width of the neural networks  
depth = 5                                           # Depth of the neural networks 
deep_layer_size = 1024*10                           # Size of the deep layer encoding the CRNs
allow_input_influence = False                       # Allow input influence in the policy
learning_rate = 1e-4                                # Learning rate for the optimizer 
hall_of_fame_size = 10                              # Size of the hall of fame  
entropy_scheduler = {                               # Entropy scheduler parameters TODO: entropy_weight should factor in batch size and expected reward scale
    'entropy_weight': 0,#50.0, 
    'entropy_update_coefficient': 0.75, 
    'entropy_schedule': 5, 
    'minimum_entropy_weight': 0,#50.0
}
risk_scheduler = {                                  # Risk scheduler parameters
    'risk': 0,#0.95, 
    'risk_update': 0.0, 
    'max_risk': 1.0, 
    'risk_schedule': 20
}
epoch_num = 10#500                                     # Number of epochs for training
render_schedule = 1                                 # Render every # of epochs
mode = {                                            # Mode of the experiment
    'style': 'logger', 
    'task': 'transients', 
    'format': 'image'
}

# Time horizon for the simulation
t_f = 200                                           # Final time for the simulation
N_t = 1000                                          # Number of time steps
time_horizon = np.linspace(0, t_f, N_t, dtype=np.float32)

# Construct the IOCRN inputs
nums = [0.5, 1.0, 1.5]
u_list = [np.array(u) for u in product(nums, repeat=p)] # list of input combinations, each input is a numpy array of shape (p,)

# Construct the IOCRN initial conditions
ic = IC(names=species_labels, values=[[0., 0., 0.]])

# Construct the weights for the performance metric
w = np.ones(N_t)
w[(len(w)//5)*4:] = w[(len(w)//5)*4:]*2
w[:(len(w)//5)] = w[:(len(w)//5)]*0.25
w = w[np.newaxis, :]

# Construct the compute reward routine
def compute_reward(state):
    r_list = [np.array([u[0]]) for u in u_list]
    x0_list = ic.get_ic(state)
    return dynamic_tracking_error(state, u_list, x0_list, time_horizon, r_list, w, norm=1, LARGE_NUMBER=1e4)

In [ ]:
# Construct parallel environments
crn_0 = crn_template.clone()
# mult_env = ParallelEnvironments([Environment(crn_0, max_added_reactions, logger=logger, logger_schedule=1) for _ in range(N)], hall_of_fame_size=hall_of_fame_size, N_CPUs=N_CPUs, logger=logger)
mult_env = SerialEnvironments([Environment(crn_0, max_added_reactions, logger=logger, logger_schedule=1) for _ in range(N)], hall_of_fame_size=hall_of_fame_size, logger=logger)

In [ ]:
# Construct the policy
encoder_attributes = {"hidden_size": width, "num_layers": depth}
structure_head_attributes = {"hidden_size": width, "num_layers": depth}
rate_head_attributes = {"hidden_size": width, "num_layers": depth}
input_influence_head_attributes = {"hidden_size": width, "num_layers": depth}
masks = {"continuous": library.get_parameter_mask(mode="continuous"), "discrete": library.get_parameter_mask(mode="discrete"), "logit": library.get_logit_mask()}
policy = AddReactionByIndex(M, K, p, encoder_attributes, deep_layer_size, structure_head_attributes, rate_head_attributes, input_influence_head_attributes, allow_input_influence=False, masks=masks, device=device)

# Construct the agent
agent = REINFORCEAgent(policy, allow_input_influence=False, logger=logger, learning_rate=learning_rate, entropy_scheduler=entropy_scheduler, risk_scheduler=risk_scheduler, device=device)
if load_flag:
    agent.policy.load_state_dict(torch.load(save_filename, map_location=device))

In [ ]:
# Construct the interfaces
observer = ExplicitObserver(reaction_library=library)
tensorizer = ExplicitTensorizer(device=device)
actuator = LibraryActuator(reaction_library=library)
stepper = IOCRNStepper()

In [ ]:
import random
seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:
# sample = 0
# mult_env.reset()
# for j in range(max_added_reactions):
#     print(f'Iteration {j}:')
#     observations = mult_env.observe(observer, tensorizer)
#     # actions = agent.act(observations, actuator)





#     actions, logP, entropy = policy(observations, mode='full') 
#     print('State before actuate:')
#     print(mult_env.envs[sample].state)
#     print('------------------------------------------------')
#     actions_new = []
#     for a in actions:
#         r = library.get_reaction(a["reaction index"])
#         r.set_parameters(a["parameters"])
#         actions_new.append(r)
#     actions = actions_new
#     # actions = [actuator.actuate(a) for a in actions]
#     # for a in actions:

#     print('State after actuate:')
#     print(mult_env.envs[sample].state)
#     print('------------------------------------------------')






    
#     # for i in range(len(actions)): 
#     #     print(actions[i])
#     # print('------------------------------------------------')
#     out = mult_env.step(actions, stepper)

# rewards = mult_env.get_reward(compute_reward)
# print(rewards)

In [ ]:
for i in tqdm(range(epoch_num)):
    mult_env.reset()
    for j in range(max_added_reactions):
        observations = mult_env.observe(observer, tensorizer)
        actions = agent.act(observations, actuator)
        out = mult_env.step(actions, stepper)
    rewards = mult_env.get_reward(compute_reward)
    print(rewards)
    agent.update(rewards, step_iteration=i)
    if i % render_schedule == 0:
        mult_env.render(rewards, n_best=1, disregarded_percentage=0.9, mode=mode)

In [ ]:
# # Training Loop     
# if train_flag:
#     for i in tqdm(range(epoch_num)):
#         mult_env.reset()
#         for j in range(max_added_reactions):
#             observations = mult_env.observe(observer, tensorizer)
#             actions = agent.act(observations, actuator)
#             out = mult_env.step(actions, stepper)
#         rewards = mult_env.get_reward(compute_reward)
#         agent.update(rewards, step_iteration=i)
#         if i % render_schedule == 0:
#             mult_env.render(rewards, n_best=1, disregarded_percentage=0.9, mode=mode)